In [1]:
import re
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()

_RUNNING_IN_NOTEBOOK = "ipykernel" in sys.modules
LOG_PATH = (
    Path(sys.argv[1])
    if not _RUNNING_IN_NOTEBOOK and len(sys.argv) > 1
    else NOTEBOOK_DIR.parent / "logs\log.txt"
)
OUT_DIR = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCK_RE = re.compile(r"^Lock:\s*(\S+)")
RUNS_RE = re.compile(r"^Runs:\s*(\d+)")
CPU_USAGE_RE = re.compile(r"^CPU Usage:\s*([\d.]+)\s*%")
DATA_RE = re.compile(
    r"threads=\s*(\d+)\s+work=\s*(\d+)\s+workload=\s*([\d.]+)\s*ms\s+"
    r"total=\s*([\d.]+)\s*ms\s+throughput=\s*([\d.]+)\s*ops/s"
)


def read_log_text(path: Path) -> str:
    raw = path.read_bytes()
    if raw.startswith(b"\xff\xfe"):
        return raw.decode("utf-16-le", errors="ignore")
    if raw.startswith(b"\xfe\xff"):
        return raw.decode("utf-16-be", errors="ignore")
    # UTF-16 without BOM shows up as interleaved NUL bytes
    if raw[:64].count(0) > 8:
        return raw.decode("utf-16-le", errors="ignore")
    return raw.decode("utf-8-sig", errors="ignore")


def load_rows(path: Path) -> pd.DataFrame:
    text = read_log_text(path)
    rows = []
    current_lock = None
    num_runs = 5
    cpu_usage_buffer: list[float] = []

    for line in text.splitlines():
        lock_match = LOCK_RE.match(line)
        if lock_match:
            current_lock = lock_match.group(1)
            cpu_usage_buffer.clear()
            continue

        runs_match = RUNS_RE.match(line)
        if runs_match:
            num_runs = int(runs_match.group(1))
            continue

        cpu_match = CPU_USAGE_RE.match(line)
        if cpu_match:
            cpu_usage_buffer.append(float(cpu_match.group(1)))
            continue

        data_match = DATA_RE.search(line)
        if data_match and current_lock:
            threads, work, workload, total, throughput = data_match.groups()
            # Each result is preceded by num_runs measurement blocks (warmups come first and are discarded)
            measurement_cpu_usage = cpu_usage_buffer[-num_runs:]
            cpu_usage_median = statistics.median(measurement_cpu_usage) if measurement_cpu_usage else float("nan")
            cpu_usage_buffer.clear()

            rows.append({
                "lock": current_lock,
                "threads": int(threads),
                "work": int(work),
                "workload_ms": float(workload),
                "total_ms": float(total),
                "throughput": float(throughput),
                "cpu_usage_percent": cpu_usage_median,
            })
    return pd.DataFrame(rows)


def plot_metric(df: pd.DataFrame, metric: str, ylabel: str, filename_prefix: str):
    """Save one separate figure per critical work size."""
    for work in sorted(df["work"].unique()):
        subset = df[df["work"] == work]

        fig, ax = plt.subplots(figsize=(8, 5))
        for lock, group in subset.groupby("lock"):
            group = group.sort_values("threads")
            ax.plot(group["threads"], group[metric], marker="o", label=lock)

        ax.set_title(f"{ylabel} vs threads (work={work})")
        ax.set_xlabel("threads")
        ax.set_ylabel(ylabel)
        ax.grid(True)
        ax.legend()

        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{filename_prefix}_work{work}.png", dpi=150)
        plt.close(fig)


def main():
    df = load_rows(LOG_PATH)
    if df.empty:
        raise SystemExit(f"No benchmark rows parsed from {LOG_PATH}")

    df.to_csv(OUT_DIR / "benchmark_results.csv", index=False)

    plot_metric(df, "workload_ms", "Workload time (ms)", "workload_vs_threads")
    plot_metric(df, "total_ms", "Total time (ms)", "total_vs_threads")
    plot_metric(df, "throughput", "Throughput (ops/s)", "throughput_vs_threads")
    plot_metric(df, "cpu_usage_percent", "CPU Usage (%)", "cpu_usage_vs_threads")

    print(f"Parsed {len(df)} rows for locks: {sorted(df['lock'].unique())}")
    print(f"Saved CSV and plots to {OUT_DIR}")

main()


Parsed 108 rows for locks: ['MCS', 'OPENMP', 'PTHREADMUTEX', 'PTHREADSPIN', 'TAS', 'TTAS']
Saved CSV and plots to c:\Users\TDA5HC\Desktop\Documents\01_CS_Master\10_ParallelComputing\MutexLock_Assignment_1\test\output
